# 01 — The data model

You now have a populated SQLite warehouse at `data/wcl.sqlite`. This
chapter explains its shape:

- the **9 base tables** (plus 6 per-round tables introduced in ADR 0007) and how they fit together,
- the **hydration pattern** (`last_fetched_at` per row),
- how to talk to the DB from Python with the helpers the package provides.

If you haven't already, run [`00_setup_and_first_crawl.ipynb`](00_setup_and_first_crawl.ipynb) first.


## 1. Working directory

Same dance as before — `cd` up to the repo root so paths resolve.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())


## 2. The base tables at a glance

| Table | Purpose | Hydratable | Notes |
|-------|---------|:---:|-------|
| `seasons` | One row per World Climbing competition season (year). | ✓ | Root entity. |
| `leagues` | League names: World Cup, World Championships, Continental Championships, … | | Static lookup; rows added as needed during hydration. |
| `season_leagues` | (season × league) pairs. Drives event discovery. | ✓ | |
| `disciplines` | Lead / speed / boulder / combined / boulder&lead. | | Static lookup. |
| `categories` | Men / Women / Youth A Male / paraclimbing classes / … | | Static lookup; carries a `gender` int. |
| `events` | Competition events: city, country, dates. | ✓ | Many per `season_league`. |
| `competitions` | (event × discipline × category) triples — one per ranking sheet. | ✓ | |
| `athletes` | Athlete profiles: name, country, height, birthday, photo URL, … | ✓ | |
| `results` | (competition × athlete × rank). | | Filled in as a side effect of hydrating `competitions`. |

**Hydratable** tables carry a `last_fetched_at` column (ISO-8601 UTC
string). NULL means *discovered but skeleton only*; a non-NULL value
is the timestamp of the last successful full fetch.

### Per-round tables (ADR 0007)

ADR 0007 added 6 more tables populated as a side effect of
hydrating `competitions`. They unpack each competition's phase
structure (qualif / semi / final), per-round and per-stage rankings,
and route-by-route ascents:

| Table | Purpose | Hydratable |
|-------|---------|:---:|
| `category_rounds` | One row per round inside a competition. | ✓ |
| `round_stages` | Sub-divisions of a round (combined sub-stages, speed-final heats); `seq=0` default stage for simple rounds. | |
| `routes` | One row per (round, route). Speed-final lanes reuse routes — keyed by IFSC route id. | ✓ |
| `round_results` | One row per (round, athlete): `rank` / `score` / `starting_group`. | |
| `stage_results` | One row per (stage, athlete). Lets every per-stage query join uniformly on `round_stage_id`. | |
| `ascents` | Wide table: one row per (stage × athlete × route) with discipline-specific columns. | |

This chapter focuses on the 9 base tables; chapter 03 shows how to
query the per-round tables via the `round_results` and `ascents`
export views.

## 3. Open the DB the right way

The package exposes a helper `open_db()` in `src/wcl_data/db/schema.py`
that:

1. opens (or creates) the SQLite file,
2. applies the schema (idempotent),
3. sets `PRAGMA foreign_keys = ON`,
4. installs `sqlite3.Row` as the row factory so you can access columns
   by name (`row["firstname"]`) instead of by index.

Always use it rather than calling `sqlite3.connect()` directly — you
get the FK pragma for free.

In [ ]:
from wcl_data.db.schema import open_db
from wcl_data.config import load_settings

settings = load_settings(require_credentials=False)
conn = open_db(settings.db_path)
print("DB:", settings.db_path)


## 4. Inspect the schema from code

`sqlite_master` is a built-in catalog table — `SELECT * FROM sqlite_master`
lets us list every object the schema created.

In [ ]:
rows = conn.execute(
    "SELECT type, name, tbl_name FROM sqlite_master "
    "WHERE type IN ('table', 'index') ORDER BY type, name"
).fetchall()

for r in rows:
    print(f"{r['type']:6s}  {r['name']:40s}  (on {r['tbl_name']})")


And here are the columns for each main table, courtesy of `PRAGMA table_info`:

In [ ]:
for tbl in ("seasons", "events", "competitions", "athletes", "results"):
    print(f"\n[{tbl}]")
    for col in conn.execute(f"PRAGMA table_info({tbl})").fetchall():
        nullable = "" if col["notnull"] else "  NULL"
        pk = "  PK" if col["pk"] else ""
        print(f"  {col['name']:24s} {col['type']:12s}{nullable}{pk}")


## 5. Row counts: skeletons vs hydrated

The hydration pattern is the conceptual core of this whole package:

- **Discovery** runs first and writes a *skeleton* row — only the
  fields we know from the parent (the `ifsc_id`, the foreign keys).
- **Hydration** later fills in the rest of the row from the entity's
  own endpoint and stamps `last_fetched_at`.

That two-phase split is what makes incremental refresh work — we can
re-discover cheaply without paying to refetch all the profile fields.

Let's count both.

In [ ]:
HYDRATABLE = ["seasons", "season_leagues", "events", "competitions", "athletes"]

print(f"{'table':16s}  {'rows':>8s}  {'hydrated':>10s}  {'skeletons':>10s}")
for tbl in HYDRATABLE:
    total = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    hydrated = conn.execute(
        f"SELECT COUNT(*) FROM {tbl} WHERE last_fetched_at IS NOT NULL"
    ).fetchone()[0]
    print(f"{tbl:16s}  {total:>8d}  {hydrated:>10d}  {total - hydrated:>10d}")


If `skeletons` is consistently zero across the board after a `pull-new`,
the warehouse is fully hydrated. A non-zero skeleton count is normal
mid-crawl or after a 4xx-error spree, and will get fixed by the next
`refresh` or `hydrate <entity>`.


## 6. The entity graph

The discovery chain is strictly one-directional:

```
       seasons
          │
          ▼
   season_leagues
          │
          ▼
       events
          │
          ▼
   competitions ──► results ◄── athletes
```

What that means concretely:

- To find new **events**, you hydrate `season_leagues`.
- To find new **competitions** and **athletes**, you hydrate
  `events` → `competitions`. (`pull-new` does both.)
- **Results** are *not* discovered separately — they're inserted as
  a side effect of hydrating a competition (the API returns the full
  ranking inside the competition payload).

`leagues`, `disciplines`, and `categories` are static lookups —
`upsert_*` methods on `Repository` write to them on demand.

## 7. The static lookups

These tables are tiny and human-readable. Let's peek at all three.

In [ ]:
print("--- leagues ---")
for r in conn.execute("SELECT id, name FROM leagues ORDER BY name"):
    print(f"  {r['id']:3d}  {r['name']}")

print("\n--- disciplines ---")
for r in conn.execute("SELECT id, name FROM disciplines ORDER BY name"):
    print(f"  {r['id']:3d}  {r['name']}")

print("\n--- categories (first 15) ---")
for r in conn.execute("SELECT id, name, gender FROM categories ORDER BY name LIMIT 15"):
    g = {0: 'male', 1: 'female'}.get(r['gender'], '?')
    print(f"  {r['id']:3d}  {r['name']:36s}  gender={g}")


### A note on the gender column

`gender` is stored as an integer (`0` = male, `1` = female) because that's
what the World Climbing API uses. The exporter views in chapter 03 translate it
back to the strings `"male"` / `"female"` for downstream consumers.

## 8. The `results` table

`results` has a composite primary key — `(competition_id, athlete_id)` —
because an athlete appears at most once per competition.

`rank` can be `NULL` (DNF, DNS, disqualified, etc.). The package's
`Repository.upsert_result` accepts that as-is.

In [ ]:
row = conn.execute("SELECT * FROM results LIMIT 1").fetchone()
print(dict(row))

null_ranks = conn.execute("SELECT COUNT(*) FROM results WHERE rank IS NULL").fetchone()[0]
total = conn.execute("SELECT COUNT(*) FROM results").fetchone()[0]
print(f"\nresults: {total} total, {null_ranks} with rank IS NULL")


## 9. A first cross-table query

Let's ask a simple question: **which cities have hosted the most events?**

That joins `events` to itself for grouping, with a guard against the
NULL `city` rows you'll see on a few older events where the location
parser couldn't extract anything.

In [ ]:
import pandas as pd

df = pd.read_sql_query(
    """
    SELECT city, country, COUNT(*) AS n_events
      FROM events
     WHERE city IS NOT NULL
     GROUP BY city, country
     ORDER BY n_events DESC
     LIMIT 10
    """,
    conn,
)
df


## 10. Close the connection

SQLite is forgiving about leaks but it's polite to clean up.

In [ ]:
conn.close()


## What's next

Chapter 02 — [`02_the_python_api.ipynb`](02_the_python_api.ipynb) —
shows how to use the package as a library: opening a `Repository`,
running the streaming `APIClient` directly, calling the fetcher
orchestrators from Python.
